In [ ]:
import sys
import numpy as np
import cantera as ct

In [2]:
ct.suppress_thermo_warnings()
pa_to_atm = 1.0/101325

In [ ]:
# mech = 'grimech30.cti'
mech = "FFCM2.yaml"
species_print = ['CH4', 'H2', 'C2H6', 'C2H4', 'C2H2']

T = range(1050, 1501, 10)
P = 1.0*ct.one_atm
Fuel = 'CH4:0.1, N2: 0.9'

output_file_name = '1bar_average_gri30.txt'

n_sim = len(T)
n_species_print = len(species_print)

t = [0.0 for i in range(n_sim)]
for i in range(n_sim):
    t[i] = 4550.0 / T[i]

output_data = [[0.0 for j in range(n_species_print+3)] for i in range(n_sim)]


print(n_sim)
print(T)
print(t)
print(n_species_print)
print(species_print)

46
[1050, 1060, 1070, 1080, 1090, 1100, 1110, 1120, 1130, 1140, 1150, 1160, 1170, 1180, 1190, 1200, 1210, 1220, 1230, 1240, 1250, 1260, 1270, 1280, 1290, 1300, 1310, 1320, 1330, 1340, 1350, 1360, 1370, 1380, 1390, 1400, 1410, 1420, 1430, 1440, 1450, 1460, 1470, 1480, 1490, 1500]
[4.333333333333333, 4.2924528301886795, 4.252336448598131, 4.212962962962963, 4.174311926605505, 4.136363636363637, 4.099099099099099, 4.0625, 4.0265486725663715, 3.991228070175439, 3.9565217391304346, 3.9224137931034484, 3.888888888888889, 3.8559322033898304, 3.823529411764706, 3.7916666666666665, 3.760330578512397, 3.7295081967213113, 3.6991869918699187, 3.6693548387096775, 3.64, 3.611111111111111, 3.5826771653543306, 3.5546875, 3.5271317829457365, 3.5, 3.4732824427480917, 3.446969696969697, 3.4210526315789473, 3.3955223880597014, 3.3703703703703702, 3.3455882352941178, 3.321167883211679, 3.2971014492753623, 3.273381294964029, 3.25, 3.226950354609929, 3.204225352112676, 3.1818181818181817, 3.1597222222222223,

In [4]:
for i in range(n_sim):
    gas = ct.Solution(mech)
    T_sim = T[i]
    t_end = t[i]
    print('Run {:d}, T = {:d} K'.format(i, T_sim))
    gas.TPX = T_sim, P, Fuel
    r = ct.IdealGasConstPressureReactor(contents=gas, energy='off', name='isothermal_reactor')
    sim = ct.ReactorNet([r])
    sim.verbose = True
    dt_max = 5.e-4

    states = ct.SolutionArray(gas, extra=['t'])
    #print('{:10s} {:10s} {:10s} {:14s}'.format(
    #    't [s]', 'T [K]', 'P [atm]', 'u [J/kg]'))

    while sim.time < t_end:
        sim.advance(sim.time + dt_max)
        states.append(r.thermo.state, t=sim.time)
        #print(sim.time)
        #print('{:10.3e} {:10.3f} {:10.5f} {:14.6f}'.format(
        #        sim.time, r.T, (r.thermo.P)*pa_to_atm, r.thermo.u))
    output_data[i][0] = T_sim
    output_data[i][1] = states.T[-1]
    output_data[i][2] = states.P[-1]*pa_to_atm
    #print(states.T[-1])
    #print(states.P[-1]*pa_to_atm)
    for j in range(n_species_print):
        output_data[i][j+3] = states.X[-1, gas.species_index(species_print[j])] * 100

Run 0, T = 1050 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:                0
Run 1, T = 1060 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:                0
Run 2, T = 1070 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:                0
Run 3, T = 1080 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:                0
Run 4, T = 1090 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:                0
Run 5, T = 1100 K
Initializing reactor network.
Reactor 0: 55 variables.
              0 sensitivity params.
Number of equations: 55
Maximum time step:    

In [5]:
np.savetxt(output_file_name, output_data)